# maxpool-reduce — ex2: swap 'max' → 'mean' in einops.reduce to build AvgPool2d alongside MaxPool2d

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `maxpool-reduce`. Running the final beacon cell reports progress against the `CNN: MaxPool as reduce` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `CNN: MaxPool as reduce` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`maxpool-reduce`** (exercise 2). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "maxpool-reduce"
DD_SUBTOPIC = "CNN: MaxPool as reduce"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## MaxPool vs AvgPool — same reduce shape, different reduction op

Ex1 built `MaxPool2d` via `einops.reduce(x, '... (h p1) (w p2) -> ... h w', 'max', p1=p, p2=p)`. The deepening move shows that swapping ONE string — `'max'` → `'mean'` — gives `AvgPool2d` over the same window. Same window geometry, same output shape, different aggregation.

```python
max_out  = reduce(x, '... (h p1) (w p2) -> ... h w', 'max',  p1=p, p2=p)
avg_out  = reduce(x, '... (h p1) (w p2) -> ... h w', 'mean', p1=p, p2=p)
```

**Shape invariant.** Both outputs have shape `(..., H/p, W/p)`. The reduction op is orthogonal to the einops pattern.

**Why these are NOT the same.** Max preserves outliers (a single bright pixel survives the pool); average dilutes them by `p²`. For ReLU activation maps where most entries are zero, max-pool keeps a sparse signal alive; avg-pool kills it. This is why classification backbones use max-pool and segmentation decoders use avg-pool.

### Exercise 2 — swap 'max' → 'mean' in einops.reduce to build AvgPool2d alongside MaxPool2d

> ```yaml
> Difficulty: 🔴🔴🔴⚪⚪
> Bloom level: Apply
> LO: Apply the einops `reduce` pattern with TWO ops on the same window geometry — `'max'` to build `MaxPool2d` and `'mean'` to build `AvgPool2d` — and verify both outputs have shape `(..., H/p, W/p)` while diverging on a contrived input where max-pool preserves outliers that avg-pool dilutes.
> Keywords: maxpool, avgpool, einops, reduce
> ```

**KCs targeted:** `einops-reduce-op-swap`, `pool-output-shape-h-over-p-w-over-p`

Implement `ex2_max_vs_avg_pool(x, p)`. Returns the tuple `(max_out, avg_out)` computed via `einops.reduce` with `'max'` and `'mean'` over the same window pattern.

Steps:
1. `max_out = reduce(x, '... (h p1) (w p2) -> ... h w', 'max', p1=p, p2=p)`.
2. `avg_out = reduce(x, '... (h p1) (w p2) -> ... h w', 'mean', p1=p, p2=p)`.
3. Return `(max_out, avg_out)`.

Inputs:
- `x`: at least 2D. Trailing axes are `(..., H, W)` with `H % p == 0` and `W % p == 0`.
- `p`: window size (int).

Output: tuple of Tensors. Both have shape `(..., H/p, W/p)`. The test will compare to `nn.MaxPool2d` / `nn.AvgPool2d` on 4D inputs.

In [ ]:
def ex2_max_vs_avg_pool(x: Tensor, p: int) -> tuple:
    """MaxPool + AvgPool over (p, p) windows via einops.reduce."""
    raise NotImplementedError()


def _test_ex2():
    import torch.nn as nn

    # === Shape invariant: both outputs have shape (..., H/p, W/p) ===
    x = t.randn(2, 3, 6, 8)
    p = 2
    max_out, avg_out = ex2_max_vs_avg_pool(x, p)
    assert max_out.shape == (2, 3, 3, 4), f'max shape wrong: {tuple(max_out.shape)}'
    assert avg_out.shape == (2, 3, 3, 4), f'avg shape wrong: {tuple(avg_out.shape)}'

    # === Max matches nn.MaxPool2d ===
    ref_max = nn.MaxPool2d(kernel_size=p)(x)
    assert t.allclose(max_out, ref_max, atol=1e-6), f'max != nn.MaxPool2d; max diff {(max_out-ref_max).abs().max().item():.2e}'

    # === Avg matches nn.AvgPool2d ===
    ref_avg = nn.AvgPool2d(kernel_size=p)(x)
    assert t.allclose(avg_out, ref_avg, atol=1e-6), f'avg != nn.AvgPool2d; max diff {(avg_out-ref_avg).abs().max().item():.2e}'

    # === Max != Avg in general — they MUST differ on real inputs ===
    diff = (max_out - avg_out).abs()
    assert diff.max() > 1e-3, f'max and avg pool must produce different outputs on random input; got max diff {diff.max().item():.4e}'

    # === The sparse-activation case: only one pixel per 2×2 is non-zero ===
    # Max preserves the bright pixel; avg dilutes it by 1/p² = 1/4.
    x_sparse = t.zeros(1, 1, 2, 2)
    x_sparse[0, 0, 0, 0] = 4.0
    m, a = ex2_max_vs_avg_pool(x_sparse, 2)
    assert m.item() == 4.0, f'max should preserve the bright pixel; got {m.item()}'
    assert a.item() == 1.0, f'avg should dilute 4.0 to 4/4=1.0; got {a.item()}'

    # === 3D input also works (no batch axis) — '...' covers it ===
    x_3d = t.randn(3, 4, 6)
    m3, a3 = ex2_max_vs_avg_pool(x_3d, 2)
    assert m3.shape == (3, 2, 3) and a3.shape == (3, 2, 3)

    # === p=1 → identity for both ===
    x_id = t.randn(2, 4, 4)
    m_id, a_id = ex2_max_vs_avg_pool(x_id, 1)
    assert t.allclose(m_id, x_id, atol=1e-6), 'p=1: max-pool is identity'
    assert t.allclose(a_id, x_id, atol=1e-6), 'p=1: avg-pool is identity'

    # === Window p=3 on a (3,3) image collapses to scalar per leading axis ===
    x_full = t.tensor([[1.0, 2.0, 3.0], [4.0, 5.0, 6.0], [7.0, 8.0, 9.0]])
    m_f, a_f = ex2_max_vs_avg_pool(x_full, 3)
    assert m_f.shape == (1, 1) and a_f.shape == (1, 1)
    assert m_f.item() == 9.0
    assert abs(a_f.item() - 5.0) < 1e-6   # mean of 1..9 is 5.0

    # === Return type ===
    ret = ex2_max_vs_avg_pool(x, p)
    assert isinstance(ret, tuple) and len(ret) == 2
    _dd_passed.add('ex2')
    print("ex2 ✓")

_test_ex2()

<details><summary>Solution</summary>

```python
import einops as _einops

def ex2_max_vs_avg_pool(x, p):
    max_out = _einops.reduce(
        x, '... (h p1) (w p2) -> ... h w', 'max', p1=p, p2=p
    )
    avg_out = _einops.reduce(
        x, '... (h p1) (w p2) -> ... h w', 'mean', p1=p, p2=p
    )
    return max_out, avg_out
```

**Same pattern, different op string.** The einops `reduce` pattern `'... (h p1) (w p2) -> ... h w'` factors each spatial axis into `(h, p)` then reduces over `p`. The op string — `'max'`, `'mean'`, `'sum'`, `'min'`, `'prod'` — picks the aggregation. Same shape contract regardless.

**Why `'...'` over an explicit `'b c'`.** The leading axes don't participate in the reduction; einops's `...` matches any number of them. This lets the same function handle 4D `(B, C, H, W)` and 3D `(C, H, W)` and even raw `(H, W)` images.

**The sparse-activation contrast** is the load-bearing test. Random inputs give a quantitative `max ≠ avg` but the magnitudes depend on the seed. A one-hot 2×2 with a 4.0 spike isolates the qualitative difference: max=4.0 (identical to input), avg=1.0 (diluted by `1/p² = 1/4`).
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex2'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex2',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()